In [37]:
import os

# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.定义工具

我们先通过一个例子来看看工具的作用。

假设我们需要一个能计算数学题的智能体。

但是，模型本身不擅长处理数学问题，如果涉及到精确计算，它往往无法给出准确答案


In [1]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    system_prompt="你是一个算术奇才。"
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="467的平方根是多少?")

for token, metadata in agent.stream(
    {"messages": [question]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我们先估算一下 \( \sqrt{467} \) 的范围：  

- \( 21^2 = 441 \)  
- \( 22^2 = 484 \)  

所以 \( 21 < \sqrt{467} < 22 \)，并且更靠近 22，因为 467 离 484 比离 441 近。  

用牛顿迭代法（或手动试算）来精确一点：  

设 \( x_0 = 21.6 \)：  
\[
x_1 = \frac{x_0 + \frac{467}{x_0}}{2} 
= \frac{21.6 + \frac{467}{21.6}}{2}
\]  
先算 \( 467 / 21.6 \)：  
\( 21.6 \times 21 = 453.6 \)，余 \( 13.4 \)；  
\( 13.4 / 21.6 \approx 0.62037 \)，所以 \( 467 / 21.6 \approx 21.62037 \)。  

于是  
\[
x_1 = \frac{21.6 + 21.62037}{2} = \frac{43.22037}{2} \approx 21.610185
\]  

再迭代一次：  
\[
x_2 = \frac{21.610185 + \frac{467}{21.610185}}{2}
\]  
先算 \( 467 / 21.610185 \)：  
\( 21.610185 \times 21.6 = 466.779996 \)，差约 0.220004，所以商 ≈ \( 21.6 + 0.01018 \approx 21.61018 \)（几乎一样）。  

所以 \( x_2 \approx \frac{21.610185 + 21.61018}{2} \approx 21.6101825 \)。  

因此：  
\[
\sqrt{467} \approx 21.610182
\]  

更精确到常用小数：  
\[
\boxed{21.61018}
\]

所以，通常涉及到精确的数学计算，我们都会通过定义工具用传统代码来计算，然后把工具交给模型去使用。


In [5]:
# 定义工具  导入tool   定西工具需要@tool 装饰器
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [6]:
# 通过装饰器定义工具名称
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [7]:
# 通过装饰器定义工具作用描述
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [16]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [10]:
tool1.invoke({"x": 467})

21.61018278497431

In [19]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

# 2.添加工具到智能体


In [43]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [22]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


467的平方根大约是 **21.61018278497431**。

这是一个无理数，所以这个结果是近似值。如果你需要更精确的结果，我可以提供更多小数位。


In [44]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我需要获取北京和杭州的天气预报来回答您的问题。让我分别查询这两个城市的天气情况。Current weather in 北京: 22 degrees C
Next 5 days: SunnyCurrent weather in 杭州: 22 degrees C
Next 5 days: Sunny根据查询结果，北京和杭州的天气情况如下：

**北京：**
- 当前温度：22°C
- 接下来5天：晴天

**杭州：**
- 当前温度：22°C  
- 接下来5天：晴天

两个城市目前温度相同，都是22°C，而且接下来5天都将是晴天天气。看起来两地都有不错的天气条件，适合出行和户外活动。

如果采用stream的updates模式，可以看到Agent执行的每一个步骤：

In [34]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()


step: model
content: [{'type': 'text', 'text': '我来帮你计算这两个数的平方根。'}, {'type': 'tool_call', 'id': 'call_00_oWChR8Xgo21mmWKW0SP9uOS9', 'name': 'square_root', 'args': {'x': 467}}, {'type': 'tool_call', 'id': 'call_01_UqzhGeRNcoSoidItA0gScaoY', 'name': 'square_root', 'args': {'x': 529}}]

step: tools
content: [{'type': 'text', 'text': '21.61018278497431'}]

step: tools
content: [{'type': 'text', 'text': '23.0'}]

step: model
content: [{'type': 'text', 'text': '计算结果如下：\n\n- **467的平方根** ≈ 21.6102\n- **529的平方根** = 23.0（因为23 × 23 = 529，所以529是完全平方数）\n\n所以：\n- √467 ≈ 21.6102\n- √529 = 23'}]



# 3.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：


In [40]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [49]:
tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://zh.hinative.com/questions/15904858',
   'title': '"我蒸蚌"是什么意思？ - HiNative',
   'content': '蒸蚌是「真棒」的意思因為“蒸蚌” 跟“真棒” 的音一模一樣，所以有一些人在寫「真棒」的時候，會故意用蒸蚌兩字，然後只是因為這樣子看起來很有趣😂 这个答案有',
   'score': 0.9998876,
   'raw_content': None},
  {'url': 'https://www.douyin.com/shipin/7266934926203832378',
   'title': 'cf蒸蚌是什么梗 - 抖音',
   'content': '#周深蒸蚌“周深蒸蚌”：粉丝玩梗的. 165. 00:00 · #周深蒸蚌“周深蒸蚌”：粉丝玩梗的 ... 你指米老鼠稍息是什么意思，蒸蚌就是吃东西（. 815. 00:00 · 你指米老鼠稍息是什么',
   'score': 0.9997086,
   'raw_content': None},
  {'url': 'https://www.facebook.com/61586315125667/videos/%E7%B1%B3%E8%80%81%E9%BC%A0%E7%9A%84%E6%84%8F%E6%80%9D/765515616568639/',
   'title': '聽到蒸蚌代表指對了   #貓狗吐槽主人#貓狗電台#米老鼠的意思#蒸蚌 ...',
   'content': '隨便指一指，聽到蒸蚌代表指對了 #貓狗吐槽主人#貓狗電台#米老鼠的意思#蒸蚌#搞笑.',
   'score': 0.99953806,
   'raw_content': None},
  {'url': 'https://www.facebook.com/61586315125667/posts/%E9%9A%A8%E4%BE%BF%E6%8C%87%E4%B8%80%E6%8C%87%E8%81%BD%E5%8

In [45]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [50]:
# 调用工具
for chunk in agent.stream(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

step: model
content: [{'type': 'text', 'text': '我来帮你搜索一下“蒸蚌”这个梗的含义和来源。'}, {'type': 'tool_call', 'id': 'call_00_OBd1FxmCVsjLjtQuDIbxYbJq', 'name': 'tavily_search', 'args': {'query': '蒸蚌是什么梗 网络用语 含义 来源'}}]

step: tools
content: [{'type': 'text', 'text': '{"query": "蒸蚌是什么梗 网络用语 含义 来源", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.douyin.com/search/%E8%92%B8%E8%9A%8C%E7%BD%91%E7%BB%9C%E8%AF%8D%E6%B1%87", "title": "蒸蚌网络词汇 - 抖音", "content": "00:16 \\"蒸蚌\\"这个短语来自于网上，它跟\\"小猫咪一样的棒\\"。小猫咪看东西来，烤菜，所以就夸它真棒。\\"真棒\\"就是用来夸人的，一般用人夸人就会说\\"真棒真棒\\"。", "score": 0.8170061, "raw_content": null}, {"url": "https://www.iesdouyin.com/search/%E7%B1%B3%E8%80%81%E9%BC%A0%E6%98%AF%E4%BB%80%E4%B9%88%E6%A2%97", "title": "米老鼠是什么梗 - 抖音", "content": "00:03 米老鼠是稍息的意思。蒸蚌是吃东西的意思。 ... 00:04 外网博主boredance的猎奇向米老鼠cos，因为装扮和行为都十分抽象，传到国内后就被不少网友剪辑整", "score": 0.60358196, "raw_content": null}, {"url": "https://eu.36kr.com/zh/p/3681255249178243", "title": "AI赋能！让宠物说人话成热门生意 - 36氪", "conte